In [1]:
# ========================================================
# CELL 1: SYSTEM ENVIRONMENT DEPENDENCY INSTALLATION
# ========================================================
import sys

# Automatically install all baseline tokenization, text processing,
# and semantic neural network scoring packages on the active container.
!pip install nltk bert-score sentencepiece pandas numpy torch
print("Colab container environment successfully configured!")

Colab container environment successfully configured!


In [2]:
# ========================================================
# CELL 2: GLOBAL RANDOM SEEDS & EXECUTION HARDWARE
# ========================================================
import os
import time
import math
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import random
import sentencepiece as spm
from nltk.translate.bleu_score import corpus_bleu
from bert_score import score as bert_score_fn

# Set rigid deterministic seeds across computational frameworks
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Verify if compute engine contains hardware GPU accelerators
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Compute Target Engine: {device}")
if device.type != 'cuda':
    print("⚠️ WARNING: GPU is inactive. Ensure you navigate to Runtime -> Change runtime type -> T4 GPU.")

Active Compute Target Engine: cuda


In [3]:
import os
import pandas as pd
from google.colab import drive

# 1. Ensure Google Drive is securely connected
print("Connecting to Google Drive...")
drive.mount('/content/drive')

# 2. Set the exact folder path found in your My Drive
DATA_DIR = "/content/drive/MyDrive/NLU_Sa_En_Datasets"
print(f"🎯 Data engine directory target successfully set to: '{DATA_DIR}'")

# 3. Dynamic split loader function
def load_and_align_split(split):
    if split == "train":
        suffix = "_10000.csv"
    elif split in ["dev", "test"]:
        suffix = "_1000.csv"
    else:
        suffix = ".csv"

    sa_path = os.path.join(DATA_DIR, f"{split}_sa{suffix}")
    en_path = os.path.join(DATA_DIR, f"{split}_en{suffix}")

    # Fallback to check standard names without numbers
    if not os.path.exists(sa_path):
        sa_path = os.path.join(DATA_DIR, f"{split}_sa.csv")
        en_path = os.path.join(DATA_DIR, f"{split}_en.csv")

    if not os.path.exists(sa_path):
        raise FileNotFoundError(f"Missing file path: {sa_path}")

    print(f"Loading {split} from: {sa_path}")
    df_sa = pd.read_csv(sa_path)

    if os.path.exists(en_path):
        df_en = pd.read_csv(en_path)
        merged = pd.merge(df_sa, df_en, on="Source_id", how="inner")
        return merged.dropna(subset=["Sentence_sa", "Sentence_en"])
    else:
        print(f"ℹ️ Target alignment hidden for [{split}]. Building blind inference structure.")
        df_sa = df_sa.dropna(subset=["Sentence_sa"])
        df_sa["Sentence_en"] = ""
        return df_sa

# 4. Load dataset splits smoothly
try:
    train_df = load_and_align_split("train")
    dev_df = load_and_align_split("dev")
    test_df = load_and_align_split("test")
    print(f"\n🎉 Success! Loaded train ({len(train_df)}), dev ({len(dev_df)}), and test ({len(test_df)}) data rows!")
except Exception as e:
    print(f"\n❌ Setup Error: {e}")

Connecting to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🎯 Data engine directory target successfully set to: '/content/drive/MyDrive/NLU_Sa_En_Datasets'
Loading train from: /content/drive/MyDrive/NLU_Sa_En_Datasets/train_sa_10000.csv
Loading dev from: /content/drive/MyDrive/NLU_Sa_En_Datasets/dev_sa_1000.csv
Loading test from: /content/drive/MyDrive/NLU_Sa_En_Datasets/test_sa_1000.csv

🎉 Success! Loaded train (10000), dev (1000), and test (1000) data rows!


In [4]:
# ========================================================
# CELL 4: TOKENIZATION & MORPHOLOGICAL HANDLING
# ========================================================
# Export raw sentences temporarily from your loaded dataframes to train your tokenizers
with open("train_sa.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(train_df["Sentence_sa"].astype(str).tolist()))
with open("train_en.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(train_df["Sentence_en"].astype(str).tolist()))

print("Training localized subword models (this may take a few seconds)...")

# Train separate configurations to automatically learn word pieces, compounding, and Sandhi rules
# We restrict vocabulary sizes to 8,000 to maximize structural learning in a low-resource setup
spm.SentencePieceTrainer.train('--input=train_sa.txt --model_prefix=sp_sa --vocab_size=8000 --pad_id=0 --unk_id=1 --bos_id=2 --eos_id=3')
spm.SentencePieceTrainer.train('--input=train_en.txt --model_prefix=sp_en --vocab_size=8000 --pad_id=0 --unk_id=1 --bos_id=2 --eos_id=3')

# Load the compiled processing models back into your workspace variables
sp_sa = spm.SentencePieceProcessor(model_file='sp_sa.model')
sp_en = spm.SentencePieceProcessor(model_file='sp_en.model')

# Strip transient text cache files from your file explorer panel
os.remove("train_sa.txt")
os.remove("train_en.txt")

print("✅ Success! Sanskrit & English subword tokenization vocabularies compiled.")
print(f"Sanskrit Vocab Size Check: {sp_sa.get_piece_size()} tokens.")
print(f"English Vocab Size Check: {sp_en.get_piece_size()} tokens.")

Training localized subword models (this may take a few seconds)...
✅ Success! Sanskrit & English subword tokenization vocabularies compiled.
Sanskrit Vocab Size Check: 8000 tokens.
English Vocab Size Check: 8000 tokens.


In [5]:
# ========================================================
# CELL 5: DATA GENERATORS & PYTORCH PIPELINE
# ========================================================
from torch.utils.data import Dataset, DataLoader

class SanskritEnglishDataset(Dataset):
    def __init__(self, df, sp_src, sp_tgt):
        # Convert pandas series columns to raw string lists
        self.src_lines = df["Sentence_sa"].astype(str).tolist()
        self.tgt_lines = df["Sentence_en"].astype(str).tolist()
        self.sp_src = sp_src
        self.sp_tgt = sp_tgt

    def __len__(self):
        return len(self.src_lines)

    def __getitem__(self, idx):
        # Build token tensors framed with structural tags: [BOS_ID (2)] + Sentence + [EOS_ID (3)]
        src_tokens = [2] + self.sp_src.encode_as_ids(self.src_lines[idx]) + [3]
        tgt_tokens = [2] + self.sp_tgt.encode_as_ids(self.tgt_lines[idx]) + [3]
        return torch.tensor(src_tokens), torch.tensor(tgt_tokens)

def sequence_padding_collate(batch):
    """
    Dynamic padding hook to pad sequences within individual batches
    to their localized maximum lengths instead of a fixed global max.
    """
    src_vectors, tgt_vectors = zip(*batch)
    src_padded = nn.utils.rnn.pad_sequence(src_vectors, batch_first=True, padding_value=0)
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_vectors, batch_first=True, padding_value=0)
    return src_padded, tgt_padded

# Instantiate optimization data loader streams
# Batch size is set to 32 to keep your T4 Colab GPU running stably
train_loader = DataLoader(SanskritEnglishDataset(train_df, sp_sa, sp_en), batch_size=32, shuffle=True, collate_fn=sequence_padding_collate)
dev_loader = DataLoader(SanskritEnglishDataset(dev_df, sp_sa, sp_en), batch_size=32, shuffle=False, collate_fn=sequence_padding_collate)

print("✅ Success! PyTorch DataLoader conduits successfully constructed.")
print(f"Total batches in Training Stream: {len(train_loader)}")
print(f"Total batches in Validation Stream: {len(dev_loader)}")

✅ Success! PyTorch DataLoader conduits successfully constructed.
Total batches in Training Stream: 313
Total batches in Validation Stream: 32


In [6]:
# ========================================================
# CELL 6: CUSTOM SEQUENTIAL ARCHITECTURE ENGINE
# ========================================================
class StructuralPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class CustomSanskritNMTTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=256, nhead=8, num_layers=4):
        super().__init__()
        self.d_model = d_model
        self.src_embedding = nn.Embedding(src_vocab, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab, d_model)
        self.positional_encoder = StructuralPositionalEncoding(d_model)

        # PyTorch standard multi-headed attention abstraction
        self.core_transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, num_encoder_layers=num_layers,
            num_decoder_layers=num_layers, dim_feedforward=512, dropout=0.1, batch_first=True
        )
        self.head_projector = nn.Linear(d_model, tgt_vocab)

    def generate_autoregressive_mask(self, sz, device):
        """
        Creates an upper triangular mask to prevent the decoder from looking ahead at
        future tokens during training.
        """
        mask = (torch.triu(torch.ones(sz, sz, device=device)) == 1).transpose(0, 1)
        return mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))

    def forward(self, src, tgt):
        # Generate padding masks to ignore padding tokens (0) during attention weights calculation
        src_padding_mask = (src == 0)
        tgt_padding_mask = (tgt == 0)

        # Embed text and scale by square root of model dimensions
        src_embeddings = self.positional_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        tgt_embeddings = self.positional_encoder(self.tgt_embedding(tgt) * math.sqrt(self.d_model))

        causal_mask = self.generate_autoregressive_mask(tgt.size(1), src.device)

        # Forward pass through our core attention matrix blocks
        transformer_representations = self.core_transformer(
            src_embeddings, tgt_embeddings, tgt_mask=causal_mask,
            src_key_padding_mask=src_padding_mask, tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )
        return self.head_projector(transformer_representations)

# Instantiate our custom structural network onto the active device
model = CustomSanskritNMTTransformer(src_vocab=8000, tgt_vocab=8000, d_model=256, nhead=8, num_layers=4).to(device)

print("✅ Success! Custom Seq2Seq Attention Transformer Model compiled.")
# Let's count parameters right here to verify our footprint size
total_params = sum(p.numel() for p in model.parameters())
print(f"Initial baseline model size checkpoint: {total_params:,} parameters.")

✅ Success! Custom Seq2Seq Attention Transformer Model compiled.
Initial baseline model size checkpoint: 11,424,576 parameters.


Note for Grading: The training cell (Cell 7) is configured to run for 40 epochs (25 at lr=0.0004 and 15 at lr=0.0001) to ensure full weight convergence and maximize the final BLEU score. Total execution time from top-to-bottom on a standard Google Colab T4 GPU is approximately 10–12 minutes.

In [7]:
# ========================================================
# CELL 7: FIXED STABLE TRAINING LOOP FOR REAL CONVERGENCE
# ========================================================
# Start with a strong, clean learning rate
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

epochs_count = 40  # 40 epochs with a stable LR will dramatically outperform 60 dead epochs
print(f"🔥 Running {epochs_count} Epochs with Stable Learning Rate Configuration...")

for epoch in range(epochs_count):
    model.train()
    cumulative_loss = 0.0
    for src_batch, tgt_batch in train_loader:
        src_batch, tgt_batch = src_batch.to(device), tgt_batch.to(device)

        tgt_input = tgt_batch[:, :-1]
        tgt_truth = tgt_batch[:, 1:]

        optimizer.zero_grad()
        predictions_logits = model(src_batch, tgt_input)

        loss = criterion(predictions_logits.reshape(-1, predictions_logits.shape[-1]), tgt_truth.reshape(-1))
        loss.backward()
        optimizer.step()
        cumulative_loss += loss.item()

    # Gently drop the learning rate only ONCE at epoch 25 to lock in fine details
    if epoch == 25:
        for param_group in optimizer.param_groups:
            param_group['lr'] = 1e-4

    current_lr = optimizer.param_groups[0]['lr']
    print(f"✅ Epoch [{epoch+1}/{epochs_count}] -> Average Loss: {cumulative_loss/len(train_loader):.5f} | LR: {current_lr}")

🔥 Running 40 Epochs with Stable Learning Rate Configuration...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


✅ Epoch [1/40] -> Average Loss: 6.19535 | LR: 0.0004
✅ Epoch [2/40] -> Average Loss: 5.38487 | LR: 0.0004
✅ Epoch [3/40] -> Average Loss: 5.01532 | LR: 0.0004
✅ Epoch [4/40] -> Average Loss: 4.74479 | LR: 0.0004
✅ Epoch [5/40] -> Average Loss: 4.51459 | LR: 0.0004
✅ Epoch [6/40] -> Average Loss: 4.30553 | LR: 0.0004
✅ Epoch [7/40] -> Average Loss: 4.11219 | LR: 0.0004
✅ Epoch [8/40] -> Average Loss: 3.93057 | LR: 0.0004
✅ Epoch [9/40] -> Average Loss: 3.75402 | LR: 0.0004
✅ Epoch [10/40] -> Average Loss: 3.58326 | LR: 0.0004
✅ Epoch [11/40] -> Average Loss: 3.41635 | LR: 0.0004
✅ Epoch [12/40] -> Average Loss: 3.25992 | LR: 0.0004
✅ Epoch [13/40] -> Average Loss: 3.10959 | LR: 0.0004
✅ Epoch [14/40] -> Average Loss: 2.97060 | LR: 0.0004
✅ Epoch [15/40] -> Average Loss: 2.83029 | LR: 0.0004
✅ Epoch [16/40] -> Average Loss: 2.70573 | LR: 0.0004
✅ Epoch [17/40] -> Average Loss: 2.58179 | LR: 0.0004
✅ Epoch [18/40] -> Average Loss: 2.46946 | LR: 0.0004
✅ Epoch [19/40] -> Average Loss: 2.35

In [8]:
# ========================================================
# CELL 8: AUTOREGRESSIVE TRANSLATION GENERATOR
# ========================================================
def execute_greedy_translation(model, raw_sanskrit_string, sp_src, sp_tgt, limit=50):
    """
    Auto-regressively predicts tokens one by one for an individual source sentence.
    """
    model.eval()
    # Tokenize input string and wrap with structural bounds: [BOS (2)] + tokens + [EOS (3)]
    src_token_ids = [2] + sp_src.encode_as_ids(raw_sanskrit_string) + [3]
    src_tensor = torch.tensor(src_token_ids, dtype=torch.long, device=device).unsqueeze(0)

    # Initialize target tracking vector with the Beginning-of-Sequence token ID
    generated_targets = [2]

    with torch.no_grad():
        for _ in range(limit):
            tgt_tensor = torch.tensor(generated_targets, dtype=torch.long, device=device).unsqueeze(0)

            # Forward pass through model to generate prediction distribution logits
            network_output = model(src_tensor, tgt_tensor)

            # Select the token ID with the maximum probability from the final sequence position
            predicted_token_id = network_output.argmax(dim=-1)[:, -1].item()
            generated_targets.append(predicted_token_id)

            # Halt auto-regressive generation immediately if model predicts the EOS tag
            if predicted_token_id == 3:
                break

    # Decode the numerical token lists back into standard English strings, stripping BOS/EOS tags
    return sp_tgt.decode(generated_targets[1:-1])

print("✅ Success! Autoregressive decoding architecture initialized.")

✅ Success! Autoregressive decoding architecture initialized.


In [9]:
# ========================================================
# CELL 9: EFFICIENCY PROFILER & EXPORT UTILITIES
# ========================================================
print("--- RUNNING COMPLIANCE PROFILING AND INFERENCE ---")

# 1. Total Model Parameters Count (Efficiency Matrix 1)
total_model_parameters = sum(p.numel() for p in model.parameters())

# 2. Total Test Set Inference Wall-Clock Timer (Efficiency Matrix 2)
inference_start_timestamp = time.time()
test_predictions_pool = []

print(f"Translating {len(test_df)} sentences from the test split...")
for index, row in test_df.iterrows():
    output_phrase = execute_greedy_translation(model, row['Sentence_sa'], sp_sa, sp_en)
    test_predictions_pool.append(output_phrase)

inference_end_timestamp = time.time()
elapsed_generation_seconds = inference_end_timestamp - inference_start_timestamp

print("\n========================================================")
print("                    EFFICIENCY PROFILE                  ")
print("========================================================")
print(f"Total Structural Model Parameters : {total_model_parameters:,}")
print(f"Total Test Set Inference Speed    : {elapsed_generation_seconds:.2f} seconds")
print("========================================================\n")

# 3. Accuracy Verification Benchmarks (Internal Validation Check)
gold_references = test_df["Sentence_en"].astype(str).tolist()

# Prepare tokens for standard unweighted NLTK corpus BLEU format
ref_bleu = [[ref.split()] for ref in gold_references]
hyp_bleu = [pred.split() for pred in test_predictions_pool]

# Check if ground truth sentences are populated (Dev validation vs Hidden private test set)
if gold_references and gold_references[0] != "":
    internal_bleu = corpus_bleu(ref_bleu, hyp_bleu)
    print(f"Calculated Baseline BLEU Score: {internal_bleu:.4f}")
else:
    print("ℹ️ Hidden private test run active: Ground truth absent. Skipping local accuracy scoring block.")

# Compute BERTScore F1 with baseline rescaling
P, R, F1 = bert_score_fn(test_predictions_pool, gold_references, lang="en", rescale_with_baseline=True)
print(f"Calculated BERTScore F1: {F1.mean().item():.4f}")

# 4. Compile Mandatory Deliverable DataFrame Structure (Rule Requirement)
submission_export_dataframe = pd.DataFrame({
    'Source_id': test_df['Source_id'].tolist(),
    'Sentence_en': test_predictions_pool
})

# Save clean structured outputs directly as submission.csv in UTF-8
submission_export_dataframe.to_csv("submission.csv", index=False, encoding="utf-8")
print("\n🎉 Success! Verified output file compiled as 'submission.csv' in your workspace folder.")

--- RUNNING COMPLIANCE PROFILING AND INFERENCE ---
Translating 1000 sentences from the test split...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(



                    EFFICIENCY PROFILE                  
Total Structural Model Parameters : 11,424,576
Total Test Set Inference Speed    : 187.41 seconds

Calculated Baseline BLEU Score: 0.0494


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Calculated BERTScore F1: 0.2452

🎉 Success! Verified output file compiled as 'submission.csv' in your workspace folder.
